In [1]:

# %% [markdown]
# ## 0) Imports

# %%
import os
import re
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional: if scipy is available, Spearman is easiest.
try:
    from scipy.stats import spearmanr
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False




# %% [markdown]
# ## 1) Configuration

In [7]:

# %%
# FULL PATH PROVIDED BY YOU
BASE_RESULTS_DIR = Path(
    "/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/calc_files/MI/testing_binning/"
    "mi_analysis_results_MI_label_entropy_all_genes_parallel"
)

# Where to write analysis outputs (summary tables + plots)
OUT_DIR = Path("_binning_robustness_analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Choose which MI column to rank by for "top genes" overlap
RANK_COLS = ["true_mi", "normalized_MI_label_entropy"]

TOP_NS = [50, 100, 250, 500, 1000]

# Scenarios we expect from the generation script
# Folder names are created by make_save_subdir(...) in the discretization script.
SCENARIO_DIRS = {
    "round_d1": "discret_round_d1",
    "bin_w0p05": "discret_bin_w0p05",
    "bin_w0p10": "discret_bin_w0p1", 
    "bin_w0p20": "discret_bin_w0p2", 
    "none": "discret_none",
}

BASELINE = "round_d1"
COMPARE_TO = ["bin_w0p05", "bin_w0p10", "bin_w0p20", "none"]

# If you want to restrict to selected datasets, list them here; otherwise auto-detect.
DATASETS_WHITELIST = None  # e.g. ["you2", "treutlein"]

print("BASE_RESULTS_DIR:", BASE_RESULTS_DIR)
print("OUT_DIR:", OUT_DIR)

# %% [markdown]
# ## 2) Helpers

# %%
def _safe_spearman(x: pd.Series, y: pd.Series) -> float:
    """Spearman correlation with fallbacks if scipy is not available."""
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    m = np.isfinite(x.values) & np.isfinite(y.values)
    if m.sum() < 3:
        return np.nan

    if HAS_SCIPY:
        return float(spearmanr(x.values[m], y.values[m]).correlation)

    # Fallback: Spearman = Pearson correlation on ranks
    xr = pd.Series(x.values[m]).rank(method="average").values
    yr = pd.Series(y.values[m]).rank(method="average").values
    return float(np.corrcoef(xr, yr)[0, 1])

def find_dataset_files(scenario_dir: Path):
    """
    Returns dict: dataset_name -> csv_path
    Accepts pattern:
      MI_label_entropy_<dataset>_<perms>_permutations_<tag>.csv
    """
    out = {}
    if not scenario_dir.exists():
        return out

    for p in scenario_dir.glob("MI_label_entropy_*_permutations_*.csv"):
        # dataset is between MI_label_entropy_ and _<perms>_permutations_
        # Example: MI_label_entropy_weinreb_50_permutations_discret_round_d1.csv
        m = re.match(r"MI_label_entropy_(.+?)_(\d+)_permutations_.*\.csv$", p.name)
        if m:
            ds = m.group(1)
            out[ds] = p
    return out

def load_one(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    # standardize columns we need
    needed = ["gene", "n_states", "true_mi", "mean_perm_mi", "95_perm_mi", "std_perm_mi", "normalized_MI_label_entropy"]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print(f"[WARN] Missing columns in {csv_path.name}: {missing}")
    return df

def topn_overlap(a: pd.DataFrame, b: pd.DataFrame, rank_col: str, n: int) -> float:
    """
    Overlap fraction of top-n genes between two dataframes.
    """
    a_top = set(a.sort_values(rank_col, ascending=False).head(n)["gene"].astype(str))
    b_top = set(b.sort_values(rank_col, ascending=False).head(n)["gene"].astype(str))
    if n == 0:
        return np.nan
    return len(a_top & b_top) / n

def merge_on_gene(df_base: pd.DataFrame, df_other: pd.DataFrame, suffix_other: str) -> pd.DataFrame:
    """
    Merge baseline and comparator by gene; add suffixes for comparator columns.
    """
    df_base_ = df_base.copy()
    df_other_ = df_other.copy()

    # keep only relevant numeric cols + gene
    keep_cols = ["gene", "n_states", "true_mi", "mean_perm_mi", "95_perm_mi", "std_perm_mi", "normalized_MI_label_entropy"]
    df_base_ = df_base_[ [c for c in keep_cols if c in df_base_.columns] ]
    df_other_ = df_other_[ [c for c in keep_cols if c in df_other_.columns] ]

    merged = df_base_.merge(df_other_, on="gene", how="inner", suffixes=("_base", f"_{suffix_other}"))
    return merged

# %% [markdown]
# ## 3) Discover datasets across scenarios

# %%
scenario_paths = {k: BASE_RESULTS_DIR / v for k, v in SCENARIO_DIRS.items()}
scenario_files = {k: find_dataset_files(p) for k, p in scenario_paths.items()}

# Determine dataset list
all_datasets = set()
for sc, mapping in scenario_files.items():
    all_datasets |= set(mapping.keys())

if DATASETS_WHITELIST is not None:
    datasets = [d for d in sorted(all_datasets) if d in set(DATASETS_WHITELIST)]
else:
    datasets = sorted(all_datasets)

print("Detected datasets:", datasets)

# Quick sanity check: which datasets are missing baseline
missing_baseline = [d for d in datasets if d not in scenario_files.get(BASELINE, {})]
if missing_baseline:
    print("[WARN] Missing baseline files for:", missing_baseline)





BASE_RESULTS_DIR: /fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/calc_files/MI/testing_binning/mi_analysis_results_MI_label_entropy_all_genes_parallel
OUT_DIR: _binning_robustness_analysis
Detected datasets: ['junker', 'treutlein', 'weinreb']


In [8]:

# %%
rows = []

for ds in datasets:
    if ds not in scenario_files.get(BASELINE, {}):
        continue

    base_path = scenario_files[BASELINE][ds]
    df_base = load_one(base_path)

    for sc in COMPARE_TO:
        if ds not in scenario_files.get(sc, {}):
            continue

        other_path = scenario_files[sc][ds]
        df_other = load_one(other_path)

        merged = merge_on_gene(df_base, df_other, sc)

        # Correlations
        for col in RANK_COLS:
            cb = f"{col}_base"
            co = f"{col}_{sc}"
            if cb in merged.columns and co in merged.columns:
                rho = _safe_spearman(merged[cb], merged[co])
            else:
                rho = np.nan

            rows.append({
                "dataset": ds,
                "baseline": BASELINE,
                "compare_to": sc,
                "metric": f"spearman_{col}",
                "value": rho,
                "n_genes_merged": merged.shape[0],
                "baseline_file": str(base_path),
                "compare_file": str(other_path),
            })

        # Top-N overlaps
        for col in RANK_COLS:
            if col not in df_base.columns or col not in df_other.columns:
                continue
            for n in TOP_NS:
                ov = topn_overlap(df_base, df_other, col, n)
                rows.append({
                    "dataset": ds,
                    "baseline": BASELINE,
                    "compare_to": sc,
                    "metric": f"top{n}_overlap_{col}",
                    "value": ov,
                    "n_genes_merged": merged.shape[0],
                    "baseline_file": str(base_path),
                    "compare_file": str(other_path),
                })

summary_df = pd.DataFrame(rows)
summary_path = OUT_DIR / "robustness_summary_metrics.tsv"
summary_df.to_csv(summary_path, sep="\t", index=False)

print("Wrote:", summary_path)
summary_df.head(10)





Wrote: _binning_robustness_analysis/robustness_summary_metrics.tsv


,dataset,baseline,compare_to,metric,value,n_genes_merged,baseline_file,compare_file
0,junker,round_d1,bin_w0p05,spearman_true_mi,0.999773,21062,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
1,junker,round_d1,bin_w0p05,spearman_normalized_MI_label_entropy,0.999773,21062,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
2,junker,round_d1,bin_w0p05,top50_overlap_true_mi,0.940000,21062,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
3,junker,round_d1,bin_w0p05,top100_overlap_true_mi,0.940000,21062,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
4,junker,round_d1,bin_w0p05,top250_overlap_true_mi,0.916000,21062,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
5,junker,round_d1,bin_w0p05,top500_overlap_true_mi,0.972000,21062,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
6,junker,round_d1,bin_w0p05,top1000_overlap_true_mi,0.982000,21062,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
7,junker,round_d1,bin_w0p05,top50_overlap_normalized_MI_label_entropy,0.940000,21062,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
8,junker,round_d1,bin_w0p05,top100_overlap_normalized_MI_label_entropy,0.940000,21062,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
9,junker,round_d1,bin_w0p05,top250_overlap_normalized_MI_label_entropy,0.916000,21062,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...


In [9]:
summary_df.tail(10)

,dataset,baseline,compare_to,metric,value,n_genes_merged,baseline_file,compare_file
134,weinreb,round_d1,none,top50_overlap_true_mi,0.440,22636,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
135,weinreb,round_d1,none,top100_overlap_true_mi,0.740,22636,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
136,weinreb,round_d1,none,top250_overlap_true_mi,0.872,22636,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
137,weinreb,round_d1,none,top500_overlap_true_mi,0.912,22636,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
138,weinreb,round_d1,none,top1000_overlap_true_mi,0.949,22636,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
139,weinreb,round_d1,none,top50_overlap_normalized_MI_label_entropy,0.440,22636,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
140,weinreb,round_d1,none,top100_overlap_normalized_MI_label_entropy,0.740,22636,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
141,weinreb,round_d1,none,top250_overlap_normalized_MI_label_entropy,0.872,22636,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
142,weinreb,round_d1,none,top500_overlap_normalized_MI_label_entropy,0.912,22636,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...
143,weinreb,round_d1,none,top1000_overlap_normalized_MI_label_entropy,0.949,22636,/fast/AG_Metzger/philipp/python_files/gene_sig...,/fast/AG_Metzger/philipp/python_files/gene_sig...


In [10]:
# %% [markdown]
# ## Robustness figures: scatter plots + top-N overlap bars + n_states diagnostics
#
# Assumes you already created:
# - summary_df (from your cell)
# - scenario_files, BASELINE, datasets
# - load_one(), merge_on_gene(), _safe_spearman()
# - OUT_DIR
#
# This cell will generate:
# 1) Scatter plots: round_d1 vs bin_w0p05 / bin_w0p10 / bin_w0p20 (true_mi)
# 2) Bar plot: Top-N overlap (true_mi) for N in {100,500,1000}
# 3) n_states diagnostics: histograms + n_states vs MI scatter
#
# Output goes to: OUT_DIR / "figures_robustness"

# %%
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

FIG_DIR = OUT_DIR / "figures_robustness"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# For your "light check" focus:
COMPARE_TO_LIGHT = ["bin_w0p05", "bin_w0p10", "bin_w0p20"]  # no 'none' needed
TOP_NS_LIGHT = [100, 500, 1000]

# ---------- Helpers (plotting) ----------

def _scatter_mi(merged, xcol, ycol, title, outpath, rho=None):
    plt.figure(figsize=(5.2, 5.0))
    plt.scatter(merged[xcol], merged[ycol], s=3, alpha=0.15)
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    if rho is not None and np.isfinite(rho):
        plt.title(f"{title}\nSpearman ρ={rho:.4f}")
    else:
        plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=250)
    plt.close()

def _bar_topn_overlap(df_metrics, dataset, outpath, rank_col="true_mi", top_ns=(100,500,1000),
                      comparisons=("bin_w0p05","bin_w0p10","bin_w0p20")):
    # Build matrix: rows=comparisons, cols=top_ns, values=overlap
    mat = []
    for comp in comparisons:
        row = []
        for n in top_ns:
            metric = f"top{n}_overlap_{rank_col}"
            sub = df_metrics[
                (df_metrics["dataset"] == dataset) &
                (df_metrics["baseline"] == BASELINE) &
                (df_metrics["compare_to"] == comp) &
                (df_metrics["metric"] == metric)
            ]
            row.append(float(sub["value"].iloc[0]) if len(sub) else np.nan)
        mat.append(row)

    mat = np.array(mat, dtype=float)
    x = np.arange(len(comparisons))
    width = 0.24

    plt.figure(figsize=(6.4, 3.8))
    for i, n in enumerate(top_ns):
        plt.bar(x + (i - (len(top_ns)-1)/2)*width, mat[:, i], width, label=f"Top {n}")

    plt.xticks(x, comparisons, rotation=0)
    plt.ylim(0, 1.02)
    plt.ylabel("Overlap fraction")
    plt.title(f"{dataset}: Top-N overlap vs {BASELINE} ({rank_col})")
    plt.legend(frameon=False, ncol=len(top_ns))
    plt.tight_layout()
    plt.savefig(outpath, dpi=250)
    plt.close()

def _hist_n_states(dfs_by_scenario, dataset, outpath, scenarios=("round_d1","bin_w0p05","bin_w0p20"), bins=60):
    plt.figure(figsize=(6.4, 4.0))
    for sc in scenarios:
        if sc not in dfs_by_scenario:
            continue
        df = dfs_by_scenario[sc]
        if "n_states" not in df.columns:
            continue
        arr = pd.to_numeric(df["n_states"], errors="coerce").dropna().values
        if arr.size == 0:
            continue
        plt.hist(arr, bins=bins, alpha=0.35, label=sc)

    plt.xlabel("n_states")
    plt.ylabel("count")
    plt.title(f"{dataset}: n_states distribution")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(outpath, dpi=250)
    plt.close()

def _scatter_n_states_vs_mi(df, dataset, scenario, outpath, mi_col="true_mi"):
    if "n_states" not in df.columns or mi_col not in df.columns:
        return
    plt.figure(figsize=(5.2, 5.0))
    plt.scatter(df["n_states"], df[mi_col], s=3, alpha=0.15)
    plt.xlabel("n_states")
    plt.ylabel(mi_col)
    plt.title(f"{dataset}: n_states vs {mi_col} ({scenario})")
    plt.tight_layout()
    plt.savefig(outpath, dpi=250)
    plt.close()

# ---------- Main: per dataset figures ----------

for ds in datasets:
    if ds not in scenario_files.get(BASELINE, {}):
        continue

    ds_dir = FIG_DIR / ds
    ds_dir.mkdir(parents=True, exist_ok=True)

    # Load baseline + available comparators
    df_base = load_one(scenario_files[BASELINE][ds])

    dfs = {BASELINE: df_base}
    for sc in COMPARE_TO_LIGHT:
        if ds in scenario_files.get(sc, {}):
            dfs[sc] = load_one(scenario_files[sc][ds])

    # 1) Scatter plots: true_mi baseline vs comparator
    for sc in COMPARE_TO_LIGHT:
        if sc not in dfs:
            continue
        merged = merge_on_gene(df_base, dfs[sc], sc)
        xcol = "true_mi_base"
        ycol = f"true_mi_{sc}"
        rho = _safe_spearman(merged[xcol], merged[ycol])
        outpath = ds_dir / f"scatter_true_mi_{BASELINE}_vs_{sc}.png"
        _scatter_mi(
            merged, xcol, ycol,
            title=f"{ds}: {BASELINE} vs {sc} (true_mi)",
            outpath=outpath,
            rho=rho
        )

    # (Optional) Scatter for normalized MI too (often redundant, but easy)
    for sc in COMPARE_TO_LIGHT:
        if sc not in dfs:
            continue
        merged = merge_on_gene(df_base, dfs[sc], sc)
        xcol = "normalized_MI_label_entropy_base"
        ycol = f"normalized_MI_label_entropy_{sc}"
        if xcol in merged.columns and ycol in merged.columns:
            rho = _safe_spearman(merged[xcol], merged[ycol])
            outpath = ds_dir / f"scatter_normMI_{BASELINE}_vs_{sc}.png"
            _scatter_mi(
                merged, xcol, ycol,
                title=f"{ds}: {BASELINE} vs {sc} (normalized MI)",
                outpath=outpath,
                rho=rho
            )

    # 2) Bar plot: Top-N overlap (true_mi)
    outpath = ds_dir / f"bar_topN_overlap_true_mi.png"
    _bar_topn_overlap(
        summary_df, ds, outpath,
        rank_col="true_mi",
        top_ns=TOP_NS_LIGHT,
        comparisons=COMPARE_TO_LIGHT
    )

    # (Optional) Bar plot: Top-N overlap (normalized MI)
    outpath = ds_dir / f"bar_topN_overlap_normMI.png"
    _bar_topn_overlap(
        summary_df, ds, outpath,
        rank_col="normalized_MI_label_entropy",
        top_ns=TOP_NS_LIGHT,
        comparisons=COMPARE_TO_LIGHT
    )

    # 3) n_states diagnostics
    _hist_n_states(
        dfs, ds,
        outpath=ds_dir / "hist_n_states.png",
        scenarios=(BASELINE, "bin_w0p05", "bin_w0p20"),
        bins=60
    )

    # n_states vs MI for baseline (main justification plot)
    _scatter_n_states_vs_mi(
        df_base, ds, BASELINE,
        outpath=ds_dir / f"scatter_n_states_vs_true_mi_{BASELINE}.png",
        mi_col="true_mi"
    )

print("✅ Robustness figures written to:", FIG_DIR)
print("Per dataset subfolders:", [p.name for p in FIG_DIR.iterdir() if p.is_dir()])


✅ Robustness figures written to: _binning_robustness_analysis/figures_robustness
Per dataset subfolders: ['junker', 'treutlein', 'weinreb']


In [11]:
# %%
print("BASE_RESULTS_DIR =", BASE_RESULTS_DIR)
print("\nScenario folder existence + number of CSVs:")
for sc, folder in SCENARIO_DIRS.items():
    p = BASE_RESULTS_DIR / folder
    n_csv = len(list(p.glob("MI_label_entropy_*.csv"))) if p.exists() else 0
    print(f"{sc:>10} -> {folder:<25} exists={p.exists()}  n_csv={n_csv}")

    if p.exists():
        ex = sorted([x.name for x in p.glob("MI_label_entropy_*.csv")])[:3]
        if ex:
            print("   examples:", ex)


BASE_RESULTS_DIR = /fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/calc_files/MI/testing_binning/mi_analysis_results_MI_label_entropy_all_genes_parallel

Scenario folder existence + number of CSVs:
  round_d1 -> discret_round_d1          exists=True  n_csv=3
   examples: ['MI_label_entropy_junker_50_permutations_discret_round_d1.csv', 'MI_label_entropy_treutlein_50_permutations_discret_round_d1.csv', 'MI_label_entropy_weinreb_50_permutations_discret_round_d1.csv']
 bin_w0p05 -> discret_bin_w0p05         exists=True  n_csv=3
   examples: ['MI_label_entropy_junker_50_permutations_discret_bin_w0p05.csv', 'MI_label_entropy_treutlein_50_permutations_discret_bin_w0p05.csv', 'MI_label_entropy_weinreb_50_permutations_discret_bin_w0p05.csv']
 bin_w0p10 -> discret_bin_w0p1          exists=True  n_csv=3
   examples: ['MI_label_entropy_junker_50_permutations_discret_bin_w0p1.csv', 'MI_label_entropy_treutlein_50_permutations_discret_bin_w0p1.csv', '

# why not without discretisation:

In [20]:
# %%
import matplotlib.pyplot as plt
import numpy as np

# quantile_table from previous cell
# index: scenario, columns: quantiles

plt.figure(figsize=(6.2, 3.6))

# Use log10 to emphasize scale differences
data = np.log10(quantile_table.values)

im = plt.imshow(data, aspect="auto", cmap="viridis")

plt.colorbar(im, label="log10(n_states / n_cells)")

plt.xticks(
    ticks=np.arange(len(quantile_table.columns)),
    labels=[str(q) for q in quantile_table.columns]
)
plt.yticks(
    ticks=np.arange(len(quantile_table.index)),
    labels=quantile_table.index
)

plt.xlabel("Quantile (fraction of genes)")
plt.ylabel("Discretization strategy")
plt.title(f"{ds}: State-space expansion across discretizations")

plt.tight_layout()
out = OUT_DIR / f"heatmap_quantiles_n_states_per_cell_{ds}.png"
plt.savefig(out, dpi=250)
plt.close()

print("Saved:", out)


Saved: _binning_robustness_analysis/heatmap_quantiles_n_states_per_cell_junker.png


# check old calcs versus new - the same?


In [12]:
# %%
from pathlib import Path
import pandas as pd
import numpy as np

# NEW results (binning tests)
NEW_DIR = Path(
    "/fast/AG_Metzger/philipp/python_files/gene_signature/"
    "mutual_information_approach/November_25_ReDo/"
    "calc_files/MI/testing_binning/"
    "mi_analysis_results_MI_label_entropy_all_genes_parallel/discret_round_d1"
)

# OLD results (original analysis, one level up)
OLD_DIR = Path(
    "/fast/AG_Metzger/philipp/python_files/gene_signature/"
    "mutual_information_approach/November_25_ReDo/"
    "calc_files/MI/"
    "mi_analysis_results_MI_label_entropy_all_genes_parallel"
)

print("NEW_DIR exists:", NEW_DIR.exists())
print("OLD_DIR exists:", OLD_DIR.exists())


NEW_DIR exists: True
OLD_DIR exists: True


In [13]:
# %%
def load_mi_csv(path):
    df = pd.read_csv(path)
    # enforce consistent sorting
    df = df.sort_values("gene").reset_index(drop=True)
    return df

def compare_mi_tables(df_old, df_new, dataset):
    out = {}

    cols_to_check = [
        "true_mi",
        "mean_perm_mi",
        "95_perm_mi",
        "std_perm_mi",
        "normalized_MI_label_entropy",
    ]

    # ensure same genes
    same_genes = np.array_equal(df_old["gene"].values, df_new["gene"].values)
    out["same_genes"] = same_genes

    if not same_genes:
        out["gene_mismatch_n"] = len(
            set(df_old["gene"]) ^ set(df_new["gene"])
        )
        return out

    for col in cols_to_check:
        if col not in df_old.columns or col not in df_new.columns:
            continue

        diff = np.abs(df_old[col].values - df_new[col].values)

        out[f"{col}_max_abs_diff"] = float(np.nanmax(diff))
        out[f"{col}_mean_abs_diff"] = float(np.nanmean(diff))
        out[f"{col}_allclose"] = bool(
            np.allclose(df_old[col].values, df_new[col].values, atol=1e-12, rtol=0)
        )

    return out


In [14]:
# %%
datasets = []

for p in OLD_DIR.glob("MI_label_entropy_*_permutations.csv"):
    ds = p.name.split("MI_label_entropy_")[1].split("_50_permutations")[0]
    datasets.append(ds)

datasets = sorted(set(datasets))
print("Datasets detected:", datasets)

results = []

for ds in datasets:
    old_path = OLD_DIR / f"MI_label_entropy_{ds}_50_permutations.csv"
    new_path = NEW_DIR / f"MI_label_entropy_{ds}_50_permutations_discret_round_d1.csv"

    if not old_path.exists() or not new_path.exists():
        print(f"[SKIP] {ds}: missing old or new file")
        continue

    df_old = load_mi_csv(old_path)
    df_new = load_mi_csv(new_path)

    cmp = compare_mi_tables(df_old, df_new, ds)
    cmp["dataset"] = ds
    results.append(cmp)

sanity_df = pd.DataFrame(results)
sanity_df


Datasets detected: ['junker', 'knoblich_day16_1', 'knoblich_day16_2', 'knoblich_day42_1', 'knoblich_day42_2', 'morris_atac', 'morris_rna', 'treutlein', 'weinreb', 'you1', 'you2', 'you3', 'you4']
[SKIP] knoblich_day16_1: missing old or new file
[SKIP] knoblich_day16_2: missing old or new file
[SKIP] knoblich_day42_1: missing old or new file
[SKIP] knoblich_day42_2: missing old or new file
[SKIP] morris_atac: missing old or new file
[SKIP] morris_rna: missing old or new file
[SKIP] you1: missing old or new file
[SKIP] you2: missing old or new file
[SKIP] you3: missing old or new file
[SKIP] you4: missing old or new file


,same_genes,true_mi_max_abs_diff,true_mi_mean_abs_diff,true_mi_allclose,mean_perm_mi_max_abs_diff,mean_perm_mi_mean_abs_diff,mean_perm_mi_allclose,95_perm_mi_max_abs_diff,95_perm_mi_mean_abs_diff,95_perm_mi_allclose,std_perm_mi_max_abs_diff,std_perm_mi_mean_abs_diff,std_perm_mi_allclose,normalized_MI_label_entropy_max_abs_diff,normalized_MI_label_entropy_mean_abs_diff,normalized_MI_label_entropy_allclose,dataset
0,True,0.0,0.0,True,0.004863,0.000448,False,0.009218,0.000889,False,0.003582,0.000315,False,0.0,0.0,True,junker
1,True,0.0,0.0,True,0.005815,0.000656,False,0.011393,0.001256,False,0.003779,0.000456,False,0.0,0.0,True,treutlein
2,True,0.0,0.0,True,0.002122,0.000158,False,0.004515,0.000303,False,0.001688,0.000111,False,0.0,0.0,True,weinreb
